# 06 — Statistical and regional analysis

Merge reviewed RHS and NFHS tables and calculate exploratory Spearman correlations. Cross-sectional association is not causation.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import spearmanr
from pathlib import Path
import pandas as pd
ROOT=Path.cwd()
if not (ROOT/"README.md").exists() and (ROOT.parent/"README.md").exists(): ROOT=ROOT.parent
RAW=ROOT/"Dataset"/"Raw_Data"; OUT=ROOT/"Dataset"/"Processed_Data"; FIG=ROOT/"Dataset"/"Notebooks"/"Visualisations"
RAW.mkdir(parents=True,exist_ok=True); OUT.mkdir(parents=True,exist_ok=True); FIG.mkdir(parents=True,exist_ok=True)
a=RAW/"rhs_state_infrastructure.csv"; b=RAW/"nfhs5_state_indicators.csv"
if not a.exists() or not b.exists(): raise FileNotFoundError("Complete reviewed RHS and NFHS inputs using notebooks 04 and 05.")
df=pd.read_csv(a).merge(pd.read_csv(b),on="state_ut",how="inner",validate="one_to_one")
print("Matched state/UTs:",df.state_ut.nunique()); display(df.head())
num=df.select_dtypes(include="number"); cols=[c for c in num if num[c].notna().sum()>=8 and num[c].nunique()>1]
corr=num[cols].corr(method="spearman"); fig,ax=plt.subplots(figsize=(11,8)); sns.heatmap(corr,annot=True,fmt=".2f",cmap="vlag",center=0,ax=ax); ax.set_title("Spearman correlations across matched states/UTs"); fig.tight_layout(); fig.savefig(FIG/"state_indicator_spearman_heatmap.png",dpi=180,bbox_inches="tight"); plt.show()
rows=[]
for i,x in enumerate(cols):
 for y in cols[i+1:]:
  pair=df[[x,y]].dropna()
  if len(pair)>=8:
   rho,p=spearmanr(pair[x],pair[y]); rows.append({"variable_1":x,"variable_2":y,"n":len(pair),"spearman_rho":rho,"p_value":p})
res=pd.DataFrame(rows).sort_values("p_value"); display(res.head(20)); res.to_csv(OUT/"state_level_spearman_correlations.csv",index=False)

State-level averages cannot describe individual households (ecological fallacy). NFHS-5 and RHS 2021–22 are not perfectly contemporaneous. Small samples and multiple testing limit inference; treat results as exploratory.